In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path
import time

In [2]:
BANK_NAME = "Woori Bank"
BANK_CODE = "WOORI"

URL = "https://spot.wooribank.com/pot/jcc"

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "X-Requested-With": "XMLHttpRequest",
    "Origin": "https://spot.wooribank.com",
    "Referer": "https://spot.wooribank.com/pot/Dream?withyou=FXDEP0005",
    "Content-Type": "application/x-www-form-urlencoded",
}

SAVE_DIR = Path("./outputs_woori_fx")
SAVE_DIR.mkdir(exist_ok=True)

In [3]:
def generate_quarter_end_dates(start_year=2004, end_year=2019):
    dates = []
    for y in range(start_year, end_year + 1):
        dates.extend([
            f"{y}.03.31",
            f"{y}.06.30",
            f"{y}.09.30",
            f"{y}.12.31",
        ])
    return dates

target_dates = generate_quarter_end_dates()

In [4]:
def fetch_woori_html(date_str, session):
    payload = {
        "withyou": "FXDEP0005",
        "__ID": "c008088",
        "KIND": "1",
        "dateParam": "0",
        "STR_BAS_DT": date_str,
        "END_BAS_DT": date_str,
        "CURRENCY": "",
        "PRD_CD": "",
        "radio_c": "on",
    }

    r = session.post(URL, data=payload, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.text

In [5]:
def clean_rate(x):
    x = str(x).strip()
    if x in ["-", ""]:
        return None
    try:
        return float(x)
    except:
        return None


def parse_woori_table(html, target_date):
    soup = BeautifulSoup(html, "lxml")
    table = soup.find("table", {"id": "exchangeRate"})

    if table is None:
        return pd.DataFrame()

    headers = [th.get_text(strip=True) for th in table.find_all("th")]
    maturity_cols = headers[2:]  # 통화, 구분 제외

    rows = []
    tbody = table.find("tbody")
    tr_list = tbody.find_all("tr")

    current_currency = None

    for tr in tr_list:
        tds = tr.find_all("td")
        vals = [td.get_text(strip=True) for td in tds]

        if len(vals) < 2:
            continue

        # 첫 줄 (통화 포함)
        if len(vals) == len(headers):
            current_currency = vals[0]
            residency = vals[1]
            rate_vals = vals[2:]
        else:
            residency = vals[0]
            rate_vals = vals[1:]

        if current_currency is None:
            continue

        for m, r in zip(maturity_cols, rate_vals):
            rows.append({
                "bank": BANK_NAME,
                "bank_code": BANK_CODE,
                "target_date": target_date,
                "currency": current_currency,
                "residency": residency,
                "maturity": m,
                "rate": clean_rate(r),
                "product_group": "외화정기예금",
                "unit": "annual % (pretax)",
            })

    return pd.DataFrame(rows)

In [6]:
session = requests.Session()

test_date = "2004.03.31"
html = fetch_woori_html(test_date, session)

df_test = parse_woori_table(html, test_date)

print(df_test.shape)
df_test.head(20)

(140, 9)


,bank,bank_code,target_date,currency,residency,maturity,rate,product_group,unit
0,Woori Bank,WOORI,2004.03.31,USD,거주자,1주일미만,0.2715,외화정기예금,annual % (pretax)
1,Woori Bank,WOORI,2004.03.31,USD,거주자,1주일이상,0.5084,외화정기예금,annual % (pretax)
2,Woori Bank,WOORI,2004.03.31,USD,거주자,1개월이상,0.7957,외화정기예금,annual % (pretax)
3,Woori Bank,WOORI,2004.03.31,USD,거주자,2개월이상,1.0153,외화정기예금,annual % (pretax)
4,Woori Bank,WOORI,2004.03.31,USD,거주자,3개월이상,1.1143,외화정기예금,annual % (pretax)
5,Woori Bank,WOORI,2004.03.31,USD,거주자,6개월이상,NaN,외화정기예금,annual % (pretax)
6,Woori Bank,WOORI,2004.03.31,USD,거주자,9개월이상,NaN,외화정기예금,annual % (pretax)
7,Woori Bank,WOORI,2004.03.31,USD,거주자,12개월이상,NaN,외화정기예금,annual % (pretax)
8,Woori Bank,WOORI,2004.03.31,USD,거주자,24개월이상,NaN,외화정기예금,annual % (pretax)
9,Woori Bank,WOORI,2004.03.31,USD,거주자,36개월이상,NaN,외화정기예금,annual % (pretax)


In [7]:
session = requests.Session()

all_parts = []
fail_log = []

for i, d in enumerate(target_dates, start=1):
    try:
        html = fetch_woori_html(d, session)
        df = parse_woori_table(html, d)

        if df.empty:
            fail_log.append({"date": d, "reason": "empty"})
        else:
            print(f"{d} OK ({len(df)} rows)")
            all_parts.append(df)

        time.sleep(0.4)

    except Exception as e:
        fail_log.append({"date": d, "reason": str(e)})

    if i % 10 == 0:
        print(f"{i}/{len(target_dates)} done")

woori_all = pd.concat(all_parts, ignore_index=True) if all_parts else pd.DataFrame()
woori_fail = pd.DataFrame(fail_log)

print("FINAL SHAPE:", woori_all.shape)

2004.03.31 OK (140 rows)
2004.06.30 OK (140 rows)
2004.09.30 OK (140 rows)
2004.12.31 OK (140 rows)
2005.03.31 OK (140 rows)
2005.06.30 OK (200 rows)
2005.09.30 OK (200 rows)
2005.12.31 OK (200 rows)
2006.03.31 OK (200 rows)
2006.06.30 OK (200 rows)
10/64 done
2006.09.30 OK (200 rows)
2006.12.31 OK (200 rows)
2007.03.31 OK (200 rows)
2007.06.30 OK (200 rows)
2007.09.30 OK (200 rows)
2007.12.31 OK (200 rows)
2008.03.31 OK (200 rows)
2008.06.30 OK (220 rows)
2008.09.30 OK (220 rows)
2008.12.31 OK (220 rows)
20/64 done
2009.03.31 OK (220 rows)
2009.06.30 OK (220 rows)
2009.09.30 OK (220 rows)
2009.12.31 OK (220 rows)
2010.03.31 OK (220 rows)
2010.06.30 OK (240 rows)
2010.09.30 OK (240 rows)
2010.12.31 OK (240 rows)
2011.03.31 OK (240 rows)
2011.06.30 OK (240 rows)
30/64 done
2011.09.30 OK (240 rows)
2011.12.31 OK (240 rows)
2012.03.31 OK (240 rows)
2012.06.30 OK (240 rows)
2012.09.30 OK (240 rows)
2012.12.31 OK (240 rows)
2013.03.31 OK (240 rows)
2013.06.30 OK (240 rows)
2013.09.30 OK (24

In [8]:
output_path = SAVE_DIR / "woori_fx_2004_2019.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    woori_all.to_excel(writer, sheet_name="raw", index=False)
    woori_fail.to_excel(writer, sheet_name="fail", index=False)

print("saved:", output_path)

saved: outputs_woori_fx\woori_fx_2004_2019.xlsx
